# Post-processing: modal filter vs CRF

Segmentations are often not perfect - especially pixelwise segmentations, which can be noisy. This necessitates 'post-processing' - applying some algorithm to the segmentations to correct errors. A common post-processing technique is the [modal filter](https://haesleinhuepf.github.io/BioImageAnalysisNotebooks/20h_segmentation_post_processing/mode_filter.html), which slides a $k \times k$ kernel across a segmented image and assigns a class to the central pixel based on the most frequent (modal) value inside the kernel. This incorporates information from neighbouring pixels and so smooths the resulting segmentation map, thereby reducing noise.

However, the modal filter is quite a strong correction, and loses lots of information we could use to correct the segmentation map, such as our classifier confidences and local colour information. This is where the Conditional Random Field, or CRF, comes in handy.


The CRF adjusts the pixel's predicted class with respect to 

1) the classifier's confidence in that pixel: less confident/lower probability = more likely to be corrected 
2) how similar that pixel is in space and colour to its neighbours with different predicted classes (very similar in colour to a neighbour with a different prediction = more likely to be corrected).

The CRF we use is [`pydensecrf`](https://github.com/lucasb-eyer/pydensecrf), which is a python wrapper around some C++ code from ['Efficient Inference in Fully Connected CRFs with Gaussian Edge Potentials' by Krähenbühl and Koltun](https://arxiv.org/abs/1210.5644). They make some clever mean field approximation and filter in bilateral space (on a permutohedral lattice), which makes it tractable to compute.


In [ ]:
from interactive_seg_backend.file_handling import load_image, load_labels
from interactive_seg_backend.configs import CRFParams
from interactive_seg_backend import FeatureConfig, TrainingConfig, featurise, train_and_apply

In [ ]:
image = load_image("../tests/data/rgb.tif")
labels = load_labels("../tests/data/rgb_labels.tif")[0]

feature_cfg = FeatureConfig()

classifier, args = 'xgb', {} #{'n_estimators': 200, 'max_depth': 10, 'max_features': 2}
default_cfg = TrainingConfig(feature_config=feature_cfg, classifier=classifier, classifier_params=args)
modal_filter_cfg = TrainingConfig(feature_config=feature_cfg, classifier=classifier, classifier_params=args, modal_filter=True, modal_filter_k=1)

crf_params = CRFParams(sxy_g=(1, 1), sxy_b=(30, 30))
crf_cfg = TrainingConfig(feature_config=feature_cfg, classifier=classifier, classifier_params=args, CRF=True, CRF_params=crf_params)

In [ ]:
feats = featurise(image, default_cfg)

Note: since the CRF uses image information to perform its post-processing, we need to supply the `image` in `train_and_apply()`

In [ ]:
preds = []
for cfg in [default_cfg, modal_filter_cfg, crf_cfg]:
    pred, _, _ = train_and_apply(feats, labels, cfg, image)
    preds.append(pred)

We plot four things here: the image with its labels (source: [Metallography and Microstructure of Aluminum and Alloys](https://vacaero.com/information-resources/metallography-with-george-vander-voort/1217-metallography-and-microstructure-of-aluminum-and-alloys.html)), the resulting segmentation with no post-processing, the segmentation with a $3\times 3$ modal filter applied and the segmentation with a CRF applied. We've got three classes: the grain, the grain boundary (white, connected) and the inclusions (white dots).

The raw classifier outputs are generally confused about what's an inclusion and what's a grain boundary (see 'No post-proc.') - it's probably fitting to large white regions as grain boundary, which causes it to mispredict on some large inclusions (and vice-versa!). The modal filter can correct some of these, but it can also remove the small grain boundaries and eat away at the grain boundary where it's thin. The CRF is able to better correct these mispredictions by integrating local colour and class information.

In [ ]:
from interactive_seg_backend.utils import apply_labels_as_overlay, PALETTE_RGB_NORM
import matplotlib.pyplot as plt
N_COLS = len(preds) + 1
fw, fh = 3, 3
fig, axs = plt.subplots(1, N_COLS, figsize=(15, 5))

img_with_labels = apply_labels_as_overlay(labels, image, PALETTE_RGB_NORM[1:])

titles = ["Image + labels", "No post-proc.", "Modal filter", "CRF"]

axs[0].imshow(img_with_labels)
axs[0].set_title(titles[0])
axs[0].axis('off')
for i, pred in enumerate(preds):
    axs[i+1].imshow(pred)
    axs[i+1].set_title(titles[i+1])
    axs[i + 1].axis('off')
        